# Modulo 3 · Lezione 2 — Gestione dei file e permessi

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vedremo oggi?

| # | Blocco | Cosa imparerai |
|---|--------|----------------|
| 1 | `cp` — copiare file e directory | Clonare senza spostare |
| 2 | `mv` — spostare e rinominare | Un comando per due usi |
| 3 | `rm` — eliminare | Con attenzione — non c'è il cestino |
| 4 | `cat`, `head`, `tail`, `less` | Leggere file senza aprirli |
| 5 | `touch` e `nano` | Creare e modificare file nel terminale |
| 6 | Permessi Linux (`rwx`) | Cosa significa ogni simbolo |
| 7 | `chmod` e `chown` | Cambiare permessi e proprietario |
| 8 | Pratica | Gestire i file del progetto ML |

> ⚠️ **Regola d'oro del terminale:** `rm` non ha il cestino. Quando elimini qualcosa, sparisce. Controlla sempre prima di eseguire.

---
## Preparare l'ambiente di demo

In [ ]:
import os, shutil
from pathlib import Path

DEMO = Path.home() / "gestion_demo"
if DEMO.exists(): shutil.rmtree(DEMO)

(DEMO / "originales").mkdir(parents=True)
(DEMO / "backups").mkdir()
(DEMO / "archivados").mkdir()

(DEMO / "originales" / "informe.txt").write_text("Report Q1\n" * 10, encoding="utf-8")
(DEMO / "originales" / "datos.csv").write_text("id,valor\n" + "1,100\n" * 20, encoding="utf-8")
(DEMO / "originales" / "script.py").write_text("print('inizio')\n" * 5, encoding="utf-8")
(DEMO / "originales" / "temporal.tmp").write_text("da eliminare", encoding="utf-8")

os.chdir(DEMO)
print(f"Demo in: {DEMO}")
!ls -R

---
## 1. `cp` — Copiare

```bash
cp origine destinazione              # copiare un file
cp origine1 origine2 directory/ # copiare più file in una directory
cp -r directory/ destinazione/     # copiare directory completa (ricorsivo)
cp -i origine destinazione           # chiedere conferma prima di sovrascrivere
cp -p origine destinazione           # preservare permessi e data
cp -u origine destinazione           # copiare solo se l'origine è più recente
```

La destinazione può essere:
- Un **nome di file** → rinomina nella destinazione.
- Una **directory esistente** → copia all'interno con lo stesso nome.
- Un **percorso nuovo** → crea il file in quel percorso.

In [ ]:
os.chdir(DEMO)

# Copiare un file
!cp originales/informe.txt backups/informe_backup.txt
print("Dopo aver copiato informe.txt:")
!ls backups/

# Copiare più file in una directory
!cp originales/datos.csv originales/script.py backups/
print("\nDopo aver copiato più file:")
!ls backups/

In [ ]:
# Copiare una directory completa (serve -r)
!cp -r originales/ backups/originales_copia/
print("Dopo aver copiato la directory:")
!ls -la backups/

# Senza -r fallisce:
import subprocess
r = subprocess.run(["cp", "originales", "backups/intento"],
                   capture_output=True, text=True)
print(f"\nSenza -r: {r.stderr.strip()}")

---
## 2. `mv` — Spostare e rinominare

`mv` fa due cose diverse a seconda della destinazione:
- Se la destinazione è un **nome diverso nella stessa directory** → **rinomina**.
- Se la destinazione è una **directory** → **sposta**.

```bash
mv vecchio.txt nuovo.txt       # rinominare
mv file.txt directory/     # spostare
mv file.txt dir/nuovo.txt   # spostare e rinominare insieme
mv *.log logs/                 # spostare tutti i .log
mv -i origine destinazione           # chiedere conferma prima di sovrascrivere
mv -n origine destinazione           # non sovrascrivere mai
```

**`mv` non ha `-r`** — le directory si spostano direttamente.

In [ ]:
os.chdir(DEMO)

# Rinominare un file
!cp originales/informe.txt informe_v1.txt
!mv informe_v1.txt informe_v2.txt
print("Rinominato:")
!ls *.txt 2>/dev/null || ls informe*.txt 2>/dev/null

# Spostare in un'altra directory
!mv informe_v2.txt archivados/
print("\nSpostato in archivados:")
!ls archivados/

# Spostare e rinominare in un solo passaggio
!cp originales/datos.csv datos_temp.csv
!mv datos_temp.csv archivados/datos_historicos.csv
print("\nSpostato e rinominato:")
!ls archivados/

---
## 3. `rm` — Eliminare

```bash
rm file.txt             # eliminare un file
rm file1 file2       # eliminarne diversi
rm *.tmp                   # eliminare tramite pattern glob
rm -r directory/          # eliminare directory e tutto il suo contenuto
rm -i file.txt          # chiedere conferma prima di eliminare
rm -f file.txt          # forzare (non chiedere se non esiste)
```

⚠️ **`rm -rf /percorso`** è uno dei comandi più pericolosi del terminale. Elimina tutto in modo ricorsivo e forzato, senza conferma. Non usarlo mai nelle directory di sistema.

**Buone pratiche:**
- Usa `-i` quando non sei sicuro.
- Usa `ls` con lo stesso glob prima di `rm`: se `ls *.tmp` mostra ciò che ti aspetti, allora `rm *.tmp` è sicuro.
- Negli script, sposta in una cartella temporanea prima di eliminare definitivamente.

In [ ]:
os.chdir(DEMO)

# Verificare prima con ls (buona pratica)
print("Cosa sto per eliminare?")
!ls originales/*.tmp

# Eliminare il file temporaneo
!rm originales/temporal.tmp
print("\nDopo rm:")
!ls originales/

# Eliminare una directory vuota (rmdir) vs con contenuto (rm -r)
!mkdir carpeta_vacia
!rmdir carpeta_vacia
print("\ncarpeta_vacia eliminata con rmdir")

# Per directory con contenuto:
!mkdir -p para_borrar/subcarpeta
!touch para_borrar/archivo.txt
print("\nPrima di rm -r:")
!ls para_borrar/

!rm -r para_borrar/
print("Dopo rm -r: directory eliminata")
!ls | grep para_borrar || echo "(non appare — eliminata correttamente)"

---
## 4. Leggere file: `cat`, `head`, `tail`, `less`

```bash
cat file.txt            # mostrare tutto il file
cat a.txt b.txt            # concatenare e mostrare due file
cat -n file.txt         # mostrare con numeri di riga

head file.txt           # prime 10 righe
head -n 5 file.txt      # prime 5 righe

tail file.txt           # ultime 10 righe
tail -n 20 file.txt     # ultime 20 righe
tail -f log.txt            # seguire il file in tempo reale (log!)

less file.txt           # lettore interattivo (q per uscire)
```

**`tail -f`** è uno dei comandi più utili in ambito IA: lo userai costantemente per monitorare log di addestramento in tempo reale.

In [ ]:
os.chdir(DEMO)

# Creare un file di log di esempio
Path("entrenamiento.log").write_text(
    "\n".join([
        f"Epoch {i:02d}/20 — loss: {2.0 - i*0.09:.4f} — val_loss: {2.1 - i*0.08:.4f}"
        for i in range(1, 21)
    ]) + "\n",
    encoding="utf-8"
)

print("=== head -n 5 (prime 5 righe) ===")
!head -n 5 entrenamiento.log

print("\n=== tail -n 5 (ultime 5 righe) ===")
!tail -n 5 entrenamiento.log

print("\n=== cat -n (con numeri di riga) ===")
!cat -n entrenamiento.log

In [ ]:
# wc — contare righe, parole, caratteri
print("=== wc (word count) ===")
!wc entrenamiento.log
# righe  parole  caratteri  nome

print("\n=== wc -l (solo righe — molto utile) ===")
!wc -l entrenamiento.log

print("\n=== wc -l su più file ===")
!wc -l originales/*.txt originales/*.csv originales/*.py

---
## 5. `touch` e `nano` — Creare e modificare

```bash
touch file.txt          # creare file vuoto (o aggiornare la data)
touch a.txt b.txt c.txt    # crearne diversi insieme

nano file.txt           # editor di testo nel terminale
```

**Scorciatoie di `nano`:**

| Scorciatoia | Azione |
|-------|--------|
| `Ctrl+O` | Salvare |
| `Ctrl+X` | Uscire |
| `Ctrl+W` | Cercare |
| `Ctrl+K` | Tagliare riga |
| `Ctrl+U` | Incollare |

> Per modificare file su server remoti senza interfaccia grafica, `nano` è l'opzione più amichevole. Esistono anche `vim` (più potente, curva di apprendimento ripida) ed `emacs`.

In [ ]:
os.chdir(DEMO)

# touch: creare file vuoti
!touch README.md .env requirements.txt
print("File creati con touch:")
!ls -la | grep -E 'README|env|requirements'

# touch aggiorna anche la data di modifica
import time
!ls -la originales/informe.txt
time.sleep(1)
!touch originales/informe.txt
print("\nDopo touch (data aggiornata):")
!ls -la originales/informe.txt

---
## 6. Permessi Linux (`rwx`)

Ogni file in Linux ha tre gruppi di permessi:

```
-rwxr-xr--  1  ana  staff  1234  gen 15  script.py
 │││└─┬─┘└─┬─┘
 │││  │    └─ altri (rest of the world)
 │││  └─ gruppo
 ││└─ proprietario (owner)
 │└─ tipo: - file, d directory, l collegamento
 └─ (abbiamo già visto il primo carattere)

Ogni gruppo ha tre bit:      r  w  x
                             │  │  └─ execute (eseguire)
                             │  └─ write (scrivere)
                             └─ read (leggere)
```

**Rappresentazione numerica (ottale):**

| Simbolo | Ottale | Significato |
|---------|-------|-------------|
| `---`   | 0     | Nessun permesso |
| `--x`   | 1     | Solo esecuzione |
| `-w-`   | 2     | Solo scrittura |
| `-wx`   | 3     | Scrittura + esecuzione |
| `r--`   | 4     | Solo lettura |
| `r-x`   | 5     | Lettura + esecuzione |
| `rw-`   | 6     | Lettura + scrittura |
| `rwx`   | 7     | Tutto |

**Permessi più comuni:**
- `644` (`rw-r--r--`) → file normale: il proprietario può leggere/scrivere, gli altri solo leggere.
- `755` (`rwxr-xr-x`) → script eseguibile: proprietario tutto, gli altri leggere ed eseguire.
- `700` (`rwx------`) → solo il proprietario può fare qualcosa (chiavi SSH, ecc.).
- `600` (`rw-------`) → file privati (`.env`, chiavi private).

In [ ]:
os.chdir(DEMO)

# Vedere i permessi attuali
print("=== Permessi attuali ===")
!ls -la originales/

# Vedere in formato ottale
print("\n=== Permessi in ottale (stat) ===")
!stat -f "%Mp%Lp %N" originales/* 2>/dev/null || \
 stat --format="%a %n" originales/* 2>/dev/null

---
## 7. `chmod` e `chown` — Cambiare permessi

```bash
# Notazione ottale (più concisa)
chmod 755 script.py        # rwxr-xr-x
chmod 644 datos.csv        # rw-r--r--
chmod 600 .env             # rw------- (solo il proprietario)

# Notazione simbolica (più leggibile)
chmod +x script.py         # aggiungere esecuzione a tutti
chmod -w archivo.txt       # togliere scrittura a tutti
chmod u+x script.py        # aggiungere esecuzione solo all'utente
chmod go-w archivo.txt     # togliere scrittura a gruppo e altri
chmod a=r archivo.txt      # tutti solo lettura

# Ricorsivo
chmod -R 755 directorio/   # applicare a tutta la directory

# Cambiare proprietario
chown utente file      # cambiare proprietario
chown utente:gruppo file # cambiare proprietario e gruppo
```

**Regola pratica:**
- Script Python/Bash che eseguirai → `chmod +x script.py`
- File `.env` con password → `chmod 600 .env`
- Chiavi SSH → `chmod 700 ~/.ssh/` e `chmod 600 ~/.ssh/id_ed25519`

In [ ]:
os.chdir(DEMO)

# Rendere eseguibile uno script
print("Prima di chmod:")
!ls -la originales/script.py

!chmod +x originales/script.py
print("\nDopo chmod +x:")
!ls -la originales/script.py

# Proteggere .env
Path(".env").write_text("API_KEY=segreto123\nDB_PASS=altra_chiave\n", encoding="utf-8")
!chmod 600 .env
print("\n.env con permessi 600:")
!ls -la .env

# Vedere i permessi in python
import stat
modo = Path("originales/script.py").stat().st_mode
print(f"\nPermessi di script.py in ottale: {oct(modo)[-3:]}")
print(f"Permessi in simbolico: {stat.filemode(modo)}")

---
## 8. Pratica — Gestire i file del progetto ML

---

### Esercizio 1 — Organizzare una directory caotica

**Consegna:** Hai una directory `caos/` con file mescolati. Organizzali automaticamente:

1. Crea `~/caos/` con questi file: `modelo.pkl`, `entrenamiento.log`, `datos.csv`, `config.yaml`, `notas.txt`, `backup.tar.gz`, `script_test.py`, `reporte.pdf`, `imagen.png`, `README.md`.
2. Scrivi uno script Python che li organizzi in sottocartelle per tipo:
   - `modelos/` → `.pkl`, `.h5`, `.pt`
   - `datos/` → `.csv`, `.json`, `.parquet`
   - `config/` → `.yaml`, `.yml`, `.toml`, `.ini`
   - `logs/` → `.log`
   - `scripts/` → `.py`, `.sh`
   - `docs/` → `.txt`, `.md`, `.pdf`
   - `otros/` → il resto
3. Verifica con `ls` ogni sottocartella.

In [ ]:
# ✏️ IL TUO CODICE QUI
from pathlib import Path
import shutil, os


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
from pathlib import Path
import shutil, os

CAOS = Path.home() / "caos"
CAOS.mkdir(exist_ok=True)

# Creare file di esempio
for nombre in ["modelo.pkl", "entrenamiento.log", "datos.csv", "config.yaml",
               "notas.txt", "backup.tar.gz", "script_test.py", "reporte.pdf",
               "imagen.png", "README.md"]:
    (CAOS / nombre).write_text(f"contenuto di {nombre}", encoding="utf-8")

# Mappa estensione → sottocartella
CATEGORIAS = {
    "modelos":  {".pkl", ".h5", ".pt", ".onnx"},
    "datos":    {".csv", ".json", ".parquet", ".tsv"},
    "config":   {".yaml", ".yml", ".toml", ".ini", ".cfg"},
    "logs":     {".log"},
    "scripts":  {".py", ".sh", ".bash"},
    "docs":     {".txt", ".md", ".pdf", ".rst"},
}

def categorizar(ext):
    for categoria, extensiones in CATEGORIAS.items():
        if ext in extensiones:
            return categoria
    return "otros"

movidos = 0
for archivo in CAOS.iterdir():
    if not archivo.is_file():
        continue
    categoria = categorizar(archivo.suffix.lower())
    destino = CAOS / categoria
    destino.mkdir(exist_ok=True)
    shutil.move(str(archivo), destino / archivo.name)
    print(f"  {archivo.name} → {categoria}/")
    movidos += 1

print(f"\nTotale spostati: {movidos}")

os.chdir(CAOS)
!ls -la
```
</details>

---
### Esercizio 2 — Copiare e versionare i file

**Consegna:** Pratica il ciclo di versionamento manuale con `cp` e `mv`:

1. In `~/ml_proyecto/`, crea `modelo.py` con una funzione `predecir(x)`.
2. Prima di modificarlo, crea un backup: `modelo_v1.py`.
3. Modifica `modelo.py`: aggiungi una funzione `evaluar(x, y)` nell'originale.
4. Prima di un'altra modifica, crea `modelo_v2.py`.
5. Sposta i backup in `modelos/checkpoints/`.
6. Elenca la directory `checkpoints/` per verificare.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
from pathlib import Path
import os

ML = Path.home() / "ml_proyecto"
os.chdir(ML)

# 1. Creare modelo.py iniziale
Path("modelo.py").write_text("""\
def predecir(x):
    return x * 2.0
""", encoding="utf-8")

# 2. Backup prima di modificare
!cp modelo.py modelo_v1.py
print("Backup v1 creato")

# 3. Modificare l'originale
contenido = Path("modelo.py").read_text()
Path("modelo.py").write_text(contenido + """
def evaluar(x, y):
    pred = predecir(x)
    return abs(pred - y)
""", encoding="utf-8")
print("modelo.py modificato")

# 4. Un altro backup
!cp modelo.py modelo_v2.py
print("Backup v2 creato")

# 5. Spostare i backup in checkpoints
!mv modelo_v1.py modelo_v2.py modelos/checkpoints/

# 6. Verificare
print("\nCheckpoints:")
!ls -la modelos/checkpoints/
```

**Nota:** Nella pratica questo si fa con Git (Modulo 2), ma capire `cp`/`mv` resta essenziale per gli script di automazione.
</details>

---
### Esercizio 3 — Leggere e analizzare log

**Consegna:** Crea un log di addestramento realistico e analizzalo:

1. Genera `entrenamiento.log` con 50 epoche: `Epoch XX/50 — loss: Y.YYYY — val_loss: Z.ZZZZ — accuracy: 0.XX`.
2. Mostra le prime cinque e le ultime cinque righe.
3. Mostra solo le righe dove l'accuracy supera 0.90 (usa `grep`).
4. Scrivi una funzione Python `resumir_log(ruta)` che legga il file e restituisca: epoca con loss minore, epoca con accuracy maggiore, e se il modello è migliorato (loss finale < loss iniziale).

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import os, re
from pathlib import Path

ML = Path.home() / "ml_proyecto"
os.chdir(ML)

# 1. Generare log di 50 epoche
import math
lineas = []
for i in range(1, 51):
    loss = 2.0 * math.exp(-i * 0.07)
    val_loss = loss * 1.05 + 0.01
    accuracy = 1 - loss / 2.5
    lineas.append(
        f"Epoch {i:02d}/50 — loss: {loss:.4f} — val_loss: {val_loss:.4f} — accuracy: {accuracy:.4f}"
    )
Path("logs/entrenamiento.log").write_text("\n".join(lineas) + "\n", encoding="utf-8")

# 2. head e tail
print("=== Prime 5 ===")
!head -n 5 logs/entrenamiento.log
print("\n=== Ultime 5 ===")
!tail -n 5 logs/entrenamiento.log

# 3. Epoche con accuracy > 0.90
print("\n=== Accuracy > 0.90 ===")
!grep -E 'accuracy: 0\.9[0-9]' logs/entrenamiento.log

# 4. Funzione Python
def resumir_log(ruta):
    patron = re.compile(
        r'Epoch (\d+)/\d+ — loss: ([\d.]+) — val_loss: [\d.]+ — accuracy: ([\d.]+)'
    )
    epocas = []
    for linea in Path(ruta).read_text(encoding="utf-8").splitlines():
        m = patron.search(linea)
        if m:
            epocas.append({
                "epoca": int(m.group(1)),
                "loss": float(m.group(2)),
                "accuracy": float(m.group(3)),
            })
    if not epocas:
        print("Nessun dato trovato")
        return

    mejor_loss = min(epocas, key=lambda e: e["loss"])
    mejor_acc  = max(epocas, key=lambda e: e["accuracy"])
    mejoro = epocas[-1]["loss"] < epocas[0]["loss"]

    print(f"\n  Epoche analizzate:   {len(epocas)}")
    print(f"  Loss minore:         Epoch {mejor_loss['epoca']:02d} → {mejor_loss['loss']:.4f}")
    print(f"  Accuracy maggiore:   Epoch {mejor_acc['epoca']:02d} → {mejor_acc['accuracy']:.4f}")
    print(f"  Il modello è migliorato?  {'Sì ✅' if mejoro else 'No ⚠️'}")
    print(f"  Loss iniziale→finale:  {epocas[0]['loss']:.4f} → {epocas[-1]['loss']:.4f}")

resumir_log("logs/entrenamiento.log")
```
</details>

---
### Esercizio 4 — Gestione dei permessi

**Consegna:** Applica i permessi corretti ai file del progetto:

1. In `~/ml_proyecto/`, rendi eseguibili tutti i file `.py` con un solo comando.
2. Proteggi il file `.env` con permessi `600`.
3. Rendi tutti i file `.csv` di sola lettura per gruppo e altri (senza modificare quelli del proprietario).
4. Scrivi una funzione Python `auditar_permisos(ruta)` che elenchi tutti i file e segni con ⚠️ quelli con permessi pericolosi (scrittura per altri: `o+w`).

In [ ]:
# ✏️ IL TUO CODICE QUI


---
### Esercizio 5 (sfida) — Script di backup automatico

**Consegna:** Scrivi la funzione Python `backup(origen, destino)` che:

1. Copia la directory `origen` in `destino/backup_YYYYMMDD_HHMMSS/` (con data e ora nel nome).
2. Esclude i file con estensione `.tmp`, `.log`, `.pyc` e le directory `__pycache__`.
3. Applica permessi `644` a tutti i file copiati e `755` alle directory.
4. Genera un file `MANIFEST.txt` nella cartella di backup con: data, origine, numero di file copiati, elenco dei file esclusi.
5. Restituisce il percorso del backup creato.

Provala facendo il backup di `~/ml_proyecto` in `~/backups`.

In [ ]:
# ✏️ IL TUO CODICE QUI
from pathlib import Path
from datetime import datetime
import shutil, stat


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
from pathlib import Path
from datetime import datetime
import shutil, stat, os

EXCLUIR_EXT = {".tmp", ".log", ".pyc"}
EXCLUIR_DIRS = {"__pycache__", ".git", ".venv", "venv"}

def backup(origen, destino):
    origen  = Path(origen).expanduser().resolve()
    destino = Path(destino).expanduser().resolve()
    destino.mkdir(parents=True, exist_ok=True)

    marca = datetime.now().strftime("%Y%m%d_%H%M%S")
    carpeta_backup = destino / f"backup_{marca}"
    carpeta_backup.mkdir()

    copiados = []
    excluidos = []

    for src in origen.rglob("*"):
        rel = src.relative_to(origen)

        # Escludere le directory proibite
        if any(parte in EXCLUIR_DIRS for parte in rel.parts):
            continue

        dst = carpeta_backup / rel

        if src.is_dir():
            dst.mkdir(parents=True, exist_ok=True)
            dst.chmod(0o755)
        elif src.is_file():
            if src.suffix.lower() in EXCLUIR_EXT:
                excluidos.append(str(rel))
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            dst.chmod(0o644)
            copiados.append(str(rel))

    # Manifest
    manifiesto = [
        f"Backup creato: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Origine:       {origen}",
        f"Destinazione:  {carpeta_backup}",
        f"File copiati: {len(copiados)}",
        f"File esclusi: {len(excluidos)}",
        "",
        "=== FILE COPIATI ===",
    ] + copiados + [
        "",
        "=== FILE ESCLUSI ===",
    ] + excluidos

    (carpeta_backup / "MANIFEST.txt").write_text(
        "\n".join(manifiesto), encoding="utf-8"
    )

    print(f"✅ Backup creato: {carpeta_backup}")
    print(f"   File copiati:  {len(copiados)}")
    print(f"   File esclusi: {len(excluidos)}")
    return carpeta_backup


ruta = backup("~/ml_proyecto", "~/backups")
print("\nContenuto del backup:")
import subprocess
subprocess.run(["ls", "-la", str(ruta)])
```
</details>

---
## Riepilogo della lezione

```bash
# ─── Copiare ───────────────────────────────────────────────
cp origine destinazione    # copiare file
cp -r dir/ destinazione/   # copiare directory
cp -i origine destinazione # chiedere conferma prima di sovrascrivere

# ─── Spostare / Rinominare ──────────────────────────────────
mv vecchio.txt nuovo.txt   # rinominare
mv file.txt dir/           # spostare
mv file.txt dir/nuovo      # spostare e rinominare

# ─── Eliminare ─────────────────────────────────────────────
rm file.txt                # eliminare file
rm -r directory/           # eliminare directory
rm -i file.txt              # con conferma

# ─── Leggere ────────────────────────────────────────────────
cat file.txt                # mostrare tutto
head -n 5 file              # prime 5 righe
tail -n 5 file               # ultime 5 righe
tail -f log.txt             # seguire in tempo reale
wc -l file                  # contare righe

# ─── Creare / Modificare ───────────────────────────────────
touch file.txt               # creare vuoto / aggiornare data
nano file.txt                # editor nel terminale

# ─── Permessi ───────────────────────────────────────────────
chmod 755 script.py        # rwxr-xr-x
chmod 644 datos.csv        # rw-r--r--
chmod 600 .env             # rw------- (privato)
chmod +x script.py         # aggiungere esecuzione
chmod -R 755 directorio/   # ricorsivo
```

## Cosa vediamo nella prossima lezione?

Nella **Lezione 3** entriamo negli **strumenti di produttività**: `grep` per cercare testo, `find` per cercare file, pipe `|` per concatenare comandi e redirezioni `>` per catturare risultati.